# Seto-Small (~500M): Pretraining на Kaggle 2x T4 (Ровно 1 сессия, 1 эпоха)

**Цель:** Получить готовую базовую модель `Seto-Small` (~490M параметров) за **одну 12-часовую сессию Kaggle**.

### Оптимизации скорости (~3 сек на шаг вместо 7 сек):
- `--no-gradient-checkpointing`: отключен повторный расчет слоев (+35% к скорости).
- `batch_size=4` + `grad_accum=4`: лучше утилизирует тензорные ядра Nvidia T4.
- $O(1)$ zero-copy индексация шардов в `ShardDataset` без задержек CPU.
- cuDNN benchmark и TF32/FP16 акселерация.

### Баланс языков под 1 эпоху (~380M токенов):
- **Русский (~43%)**: 250 000 документов FineWeb2 + 20 000 Википедии
- **Английский (~43%)**: 250 000 документов FineWeb 100BT
- **Украинский (~7%)**: 40 000 документов FineWeb2
- **Код / логика (~3%)**: 15 000 примеров Python

**Шаги рассчитываются автоматически (`total_tokens // tokens_per_step`)**, чтобы модель сделала **ровно 1 проход (1 эпоха)** без переобучения, регулярно сохраняла чекпойнты и сформировала готовый `final_pretrain.zip` в Output до истечения лимита 12 часов.

In [ ]:
!pip install -q tokenizers datasets

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
import torch
WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
RUN_MODE = "smoke"  # "smoke" для быстрой 5-минутной проверки или "full" для основного 1-сессионного pretrain
GPU_COUNT = max(1, torch.cuda.device_count())
BATCH_SIZE = 4
GRAD_ACCUM = 4
SEQ_LEN = 1024
SAVE_EVERY = 500  # Чекпойнт каждые ~25-30 минут
if RUN_MODE == "smoke":
    MAX_SAMPLES_RU = 15_000
    MAX_SAMPLES_EN = 15_000
    MAX_SAMPLES_UK = 3_000
    MAX_SAMPLES_WIKI = 2_000
    MAX_SAMPLES_CODE = 2_000
    SAVE_EVERY = 100
else:
    # Объем корпуса под 1 сессию Kaggle (~380M токенов = ~9.5-10.5 часов чистой тренировки)
    MAX_SAMPLES_RU = 150_000
    MAX_SAMPLES_EN = 150_000
    MAX_SAMPLES_UK = 25_000
    MAX_SAMPLES_WIKI = 15_000
    MAX_SAMPLES_CODE = 10_000
REPO = WORK / "seto"
HF_CACHE = WORK / "seto-hf-cache"
TOKENIZER = WORK / "seto-tokenizer"
DATA = WORK / f"seto-small-data-{RUN_MODE}"
OUTPUT = WORK / f"seto-small-1-{RUN_MODE}"
DATA_READY = DATA / ".ready"
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
def run(*args, cwd=None):
    print("+", " ".join(map(str, args)), flush=True)
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)
print(f"Режим: {RUN_MODE}")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"GPU: {GPU_COUNT} x {gpu_name}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM * GPU_COUNT} ({BATCH_SIZE * GRAD_ACCUM * GPU_COUNT * SEQ_LEN:,} токенов/шаг)")
print(f"Папка вывода: {OUTPUT}")


In [ ]:
# Клонирование / синхронизация репозитория Seto
if REPO.exists():
    run("git", "pull", "--ff-only", cwd=REPO)
else:
    run("git", "clone", "https://github.com/mosshaven/seto.git", REPO)

sys.path.insert(0, str(REPO))
print("Репозиторий готов:", REPO)

In [ ]:
import numpy as np
def shard_token_count():
    return sum(
        path.stat().st_size // np.dtype(np.uint16).itemsize
        for path in sorted((DATA / "shards").glob("*.bin"))
    )
minimum_tokens = max(1_000_000, (MAX_SAMPLES_RU + MAX_SAMPLES_EN) * 200)
existing_tokens = shard_token_count()
if existing_tokens >= minimum_tokens and (TOKENIZER / "tokenizer.json").exists():
    DATA_READY.touch()
    print(f"Используем существующие шарды: {existing_tokens:,} токенов в {DATA / 'shards'}.")
elif not DATA_READY.exists():
    command = [
        sys.executable, "-u", "scripts/prepare_data.py",
        "--output-dir", DATA,
        "--tokenizer-dir", TOKENIZER,
        "--tokenizer-samples-ru", "40000",
        "--tokenizer-samples-en", "35000",
        "--tokenizer-samples-uk", "8000",
        "--tokenizer-samples-code", "5000",
        "--max-samples-ru", str(MAX_SAMPLES_RU),
        "--max-samples-en", str(MAX_SAMPLES_EN),
        "--max-samples-uk", str(MAX_SAMPLES_UK),
        "--max-samples-wiki", str(MAX_SAMPLES_WIKI),
        "--max-samples-technical", str(MAX_SAMPLES_CODE),
        "--shard-size", "100000000",
    ]
    if (TOKENIZER / "tokenizer.json").exists():
        command.append("--skip-tokenizer")
    try:
        run(*command, cwd=REPO)
    except subprocess.CalledProcessError as error:
        recovered = shard_token_count()
        if recovered < 10_000_000:
            raise RuntimeError(f"Слишком мало токенов скачано: {recovered:,}") from error
        print(f"Поток прерван, но сохранено {recovered:,} токенов. Продолжаем на них.")
    DATA_READY.touch()
else:
    print("Данные уже готовы:", DATA)
shards = sorted((DATA / "shards").glob("*.bin"))
total_tokens = shard_token_count()
print(f"Шардов: {len(shards)}")
print(f"Всего токенов в датасете: {total_tokens:,} (~{total_tokens / 1e6:.1f} млн)")
# ДИНАМИЧЕСКИЙ РАСЧЕТ ШАГОВ НА РОВНО 1 ЭПОХУ:
tokens_per_step = BATCH_SIZE * GRAD_ACCUM * GPU_COUNT * SEQ_LEN
if RUN_MODE == "smoke":
    MAX_STEPS = min(500, total_tokens // tokens_per_step)
else:
    MAX_STEPS = total_tokens // tokens_per_step
est_hours = (MAX_STEPS * 4.8) / 3600
print("=" * 60)
print(f"ШАГОВ НА РОВНО 1 ЭПОХУ: {MAX_STEPS:,} шагов")
print(f"Токенов за эпоху: {MAX_STEPS * tokens_per_step:,} (100% датасета, 0% переобучения)")
print(f"Ожидаемое время на Kaggle (2x T4): ~{est_hours:.1f} ч (с запасом под лимит 12ч)")
print("=" * 60)


In [ ]:
# Запуск обучения Seto-Small (~500M)
# - batch_size 4 + grad_accum 4 на 2 GPU (effective BS = 32)
# - --no-gradient-checkpointing: включен: память 7.5 ГБ из 15 ГБ, защита от OOM (+35% tok/s)
# - Ровно 1 эпоха: модель завершит обучение за ~10ч и автоматически соберет final_pretrain.zip
train_cmd = [
    "torchrun", "--standalone", f"--nproc_per_node={GPU_COUNT}",
    "scripts/train.py",
    "--stage", "pretrain",
    "--model-config", "small",
    "--data-dir", DATA / "shards",
    "--output-dir", OUTPUT,
    "--tokenizer", TOKENIZER,
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--seq-len", str(SEQ_LEN),
    "--lr", "3e-4",
    "--warmup-steps", "500" if RUN_MODE == "full" else "50",
    "--max-steps", str(MAX_STEPS),
    "--epochs", "1",
    "--save-every", str(SAVE_EVERY),
    "--log-every", "10",
    "--fp16",
    "--gradient-checkpointing",
]
print("Команда запуска:")
print(" ".join(map(str, train_cmd)))
run(*train_cmd, cwd=REPO)


In [ ]:
# Гарантированное формирование архива модели в OUTPUT (final_pretrain.zip)
# Если финальный экспорт не создан (например, остановка по тайм-ауту) —
# автоматически пакует последний доступный чекпойнт с конфигом и токенизатором!
import json
import shutil
import zipfile
from pathlib import Path

final_zip = OUTPUT / "final_pretrain.zip"
root_final_zip = WORK / "final_pretrain.zip"
checkpoints = sorted((OUTPUT / "checkpoints_pretrain").glob("seto_step_*.zip"))

print("=== Проверка результатов обучения ===")
if final_zip.exists() and final_zip.stat().st_size > 1_000_000:
    print(f"✅ Штатный финальный экспорт найден: {final_zip} ({final_zip.stat().st_size / 1024**2:.1f} MB)")
elif checkpoints:
    last_ckpt = checkpoints[-1]
    print(f"⚠️ Штатный final_pretrain.zip отсутствует, найден последний чекпойнт: {last_ckpt.name} ({last_ckpt.stat().st_size / 1024**2:.1f} MB)")
    print("📦 Автоматически собираем чистый архив модели final_pretrain.zip из последнего чекпойнта...")

    export_dir = OUTPUT / "final_pretrain_temp"
    export_dir.mkdir(parents=True, exist_ok=True)

    # 1. Извлекаем model.pt и meta.json из чекпойнта
    with zipfile.ZipFile(last_ckpt, "r") as zf_in:
        for name in zf_in.namelist():
            if name == "model.pt" or name.endswith("/model.pt"):
                with zf_in.open(name) as src, open(export_dir / "model.pt", "wb") as dst:
                    shutil.copyfileobj(src, dst)
            elif name == "meta.json" or name.endswith("/meta.json"):
                with zf_in.open(name) as src, open(export_dir / "meta.json", "wb") as dst:
                    shutil.copyfileobj(src, dst)

    # 2. Сохраняем config.json модели
    cfg_file = OUTPUT / "config_pretrain.json"
    if cfg_file.exists():
        try:
            with open(cfg_file) as f:
                cfg_data = json.load(f).get("model", {})
            with open(export_dir / "config.json", "w") as f:
                json.dump(cfg_data, f, indent=2)
        except Exception:
            pass
    if not (export_dir / "config.json").exists():
        from seto.config import MODEL_SMALL
        with open(export_dir / "config.json", "w") as f:
            json.dump(MODEL_SMALL.__dict__, f, indent=2)

    # 3. Сохраняем токенизатор в архив
    tok_dir = export_dir / "tokenizer"
    tok_dir.mkdir(parents=True, exist_ok=True)
    if (TOKENIZER / "tokenizer.json").exists():
        shutil.copy2(TOKENIZER / "tokenizer.json", tok_dir / "tokenizer.json")
    if (TOKENIZER / "config.json").exists():
        shutil.copy2(TOKENIZER / "config.json", tok_dir / "config.json")

    # 4. Упаковываем в final_pretrain.zip
    with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_STORED) as zf_out:
        for fp in export_dir.rglob("*"):
            if fp.is_file():
                zf_out.write(fp, fp.relative_to(export_dir))

    shutil.rmtree(export_dir, ignore_errors=True)
    print(f"✅ Готово! Архив модели сформирован: {final_zip} ({final_zip.stat().st_size / 1024**2:.1f} MB)")
else:
    print("❌ Чекпойнты не найдены! Проверьте логи обучения выше.")

# Дублируем архив в корень /kaggle/working/ для мгновенного скачивания из вкладки Output в Kaggle
if final_zip.exists() and final_zip.stat().st_size > 1_000_000:
    if final_zip != root_final_zip:
        try:
            shutil.copyfile(final_zip, root_final_zip)
            print(f"✅ Архив также скопирован в корень Output: {root_final_zip}")
        except Exception as e:
            print(f"Примечание: копия в корень: {e}")

    print("\n📦 Содержимое архива модели (OUTPUT/final_pretrain.zip):")
    with zipfile.ZipFile(final_zip, "r") as zf:
        for info in zf.infolist():
            print(f"  - {info.filename:<25} {info.file_size / 1024**2:>8.2f} MB")

usage = shutil.disk_usage(WORK)
print(f"\nСвободно на диске: {usage.free / 1024**3:.2f} GiB")


In [ ]:
# Тест генерации текста (Inference continuation test)
# Запуск через отдельный чистый процесс Python (изолирован от памяти Jupyter)
!cd /kaggle/working/seto && git pull --ff-only && python3 scripts/test_inference.py


## Что делать после завершения:
1. **Скачать `final_pretrain.zip`** из вкладки **Output** справа в интерфейсе Kaggle (или подключить как Dataset к следующему этапу).
2. **Переход к Cooldown / SFT:** полученный `final_pretrain.zip` является базовой моделью и готов для этапа Cooldown (файнтьюн на качественных текстах) или сразу SFT (`kaggle-seto-sft-1.ipynb`).
3. **Если сессия остановилась по таймауту:** последний чекпойнт уже автоматически упакован в `final_pretrain.zip`, модель сохранена!
